# Methods paper — Diagnostics Notebook 01

**Purpose.** Quantify *why* the original multimodal-BD pipeline reported 82.6% macro-F1 while passing the standard
validation battery. Every experiment here is a diagnostic; nothing is tuned to improve performance.

**Part A** runs on the exact arrays behind the reported numbers (`multi_source_results/validation_data.npz`), if present.
**Part B** rebuilds the pipeline from the raw datasets, *with provenance metadata* (subject IDs, window indices, source corpus),
so leakage and circularity can be measured.

**How to run.** Runtime → Change runtime type → GPU (only the BERT step needs it). Then Runtime → Run all.
If a cell stops with a `CONFIG NEEDED` message, fix the setting it names in cell 0 and re-run from there.

**What to send back.** The block printed by the final cell (between the `=== PASTE BACK ===` markers),
plus `results.json` from `OUT_DIR` if anything looks off.

Expected runtime: ~10 min for Part A, 30–60 min for Part B (BERT embeddings are cached after the first run).

In [ ]:
#@title 0 · Setup & configuration
import os, json, pickle, time, warnings, re
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')


WESAD_SOURCE = 'complete'
SMOKE_TEST = os.environ.get('BD_SMOKE_TEST') == '1'   # internal: synthetic-data test of this notebook

if not SMOKE_TEST:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        print('Not running in Colab: set the paths below to your local copies.')

# ---- Paths: copied from the original notebooks. Note the trailing spaces in some folder names. ----
DEV = '/content/drive/MyDrive/Research on BD NYU /development'
PATHS = {
    'WESAD':          f'/content/drive/MyDrive/Research on BD NYU /development/final_modelling/datasets/raw/WESAD/WESAD',
    'OBF':            f'/content/drive/MyDrive/Research on BD NYU /development/final_modelling/datasets/raw/OBF-Psychiatric Dataset',
    'KAGGLE_BD_CSV':  f'/content/drive/MyDrive/Research on BD NYU /development/final_modelling/datasets/raw/Kaggle_BD /bipolar_dataset.csv',
    'MULTICLASS_CSV': f'/content/drive/MyDrive/Research on BD NYU /development/final_modelling/datasets/raw/multi class depression dataset/dataset.csv',
    # Saved by the original run; read by control_experimentation_multi_fusion.ipynb
    'ORIGINAL_RESULTS_DIR': '/content/drive/MyDrive/Research on BD NYU/multi_source_results',
}
# Written to your own Drive so it works even if the shared folder is read-only for you.
OUT_DIR = '/content/drive/MyDrive/bd_methods_paper/run_01'

# ---- OBF column names (auto-detected in Part B; set explicitly if detection fails) ----
OBF_ID_COL = None      # column identifying the person (e.g. 'number')
OBF_DIAG_COL = None    # column with the clinical diagnosis / group

# ---- Experiment sizes ----
SEED = 42
N_SHUFFLES = 20        # shuffle trials per condition (Part A)
N_CV_REPEATS = 5       # repeats of 5-fold CV
N_BUILDS = 10          # independent re-draws of the label-conditional pairing (Part B waterfall)
WATERFALL_CV_REPEATS = 3
BERT_BATCH = 64

if SMOKE_TEST:
    base = os.environ['BD_SMOKE_DIR']
    PATHS = {'WESAD': f'{base}/WESAD', 'OBF': f'{base}/OBF', 'KAGGLE_BD_CSV': f'{base}/kaggle.csv',
             'MULTICLASS_CSV': f'{base}/multiclass.csv', 'ORIGINAL_RESULTS_DIR': f'{base}/multi_source_results'}
    OUT_DIR = f'{base}/out'
    N_SHUFFLES, N_CV_REPEATS, N_BUILDS, WATERFALL_CV_REPEATS = 3, 2, 2, 1

CACHE_DIR = f'{OUT_DIR}/cache'
os.makedirs(CACHE_DIR, exist_ok=True)

# 'as_run'   = the 11-subject copy the original pipeline actually used
# 'complete' = all 15 subjects, extracted from WESAD.zip
WESAD_SOURCE = 'as_run'
PATHS['WESAD'] = {'as_run':   f'{DEV}/final_modelling/datasets/raw/WESAD/WESAD',
                  'complete': '/content/WESAD_extracted/WESAD'}[WESAD_SOURCE]

OUT_DIR   = f'/content/drive/MyDrive/bd_methods_paper/run_{WESAD_SOURCE}'
CACHE_DIR = '/content/drive/MyDrive/bd_methods_paper/cache'   # shared, so BERT isn't recomputed
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
print('OUT_DIR:', OUT_DIR)
print('CACHE_DIR:', CACHE_DIR)
for k, p in PATHS.items():
    print(f'  {k:22s} exists={os.path.exists(p)}  {p}')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (train_test_split, StratifiedKFold, StratifiedGroupKFold,
                                     GroupKFold, StratifiedShuffleSplit)
from sklearn.metrics import f1_score, adjusted_rand_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV

CLASS_NAMES = ['Euthymia', 'Depression', 'Mania']
RESULTS = {'meta': {'started': time.strftime('%Y-%m-%d %H:%M'), 'smoke_test': SMOKE_TEST}}

def save_results():
    def conv(o):
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, np.ndarray): return o.tolist()
        return str(o)
    with open(f'{OUT_DIR}/results.json', 'w') as f:
        json.dump(RESULTS, f, indent=2, default=conv)

def rf(seed=SEED):
    """Identical hyperparameters to the original pipeline."""
    return RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=seed, n_jobs=-1)

def macro_f1(y, p):
    return float(f1_score(y, p, average='macro'))

def original_split(y):
    """The exact split used for every reported number: stratified 80/20, random_state=42."""
    return train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=42)

def split_f1(X, y, model_seed=SEED):
    tr, te = original_split(y)
    return macro_f1(y[te], rf(model_seed).fit(X[tr], y[tr]).predict(X[te]))

def _folds(y, groups, seed):
    if groups is None:
        return StratifiedKFold(5, shuffle=True, random_state=seed).split(np.zeros(len(y)), y)
    k = min(5, len(np.unique(groups)))
    return StratifiedGroupKFold(k, shuffle=True, random_state=seed).split(np.zeros(len(y)), y, groups)

def cv_f1(X, y, groups=None, n_repeats=N_CV_REPEATS, seed=SEED):
    """Repeated 5-fold CV. Out-of-fold predictions are pooled per repeat; returns (mean, sd) over repeats."""
    scores = []
    for r in range(n_repeats):
        oof = np.empty_like(y)
        for tr, te in _folds(y, groups, seed + r):
            oof[te] = rf(seed + r).fit(X[tr], y[tr]).predict(X[te])
        scores.append(macro_f1(y, oof))
    return float(np.mean(scores)), float(np.std(scores))

def perm_index(n, rng, y=None):
    """Row permutation. With y: permute only within each class (label-preserving shuffle)."""
    if y is None:
        return rng.permutation(n)
    idx = np.arange(n)
    for c in np.unique(y):
        pos = np.where(y == c)[0]
        idx[pos] = rng.permutation(pos)
    return idx

def ece(y_true, y_prob, n_bins=10):
    """Confidence ECE, identical to compute_calibration_metrics() in the original control notebook."""
    conf = y_prob.max(1); acc = (y_prob.argmax(1) == y_true).astype(float)
    edges = np.linspace(0, 1, n_bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += abs(conf[m].mean() - acc[m].mean()) * m.mean()
    return float(e)

def fmt(m, s=None):
    return f'{m:.3f}' if s is None else f'{m:.3f} ± {s:.3f}'

print('OUT_DIR:', OUT_DIR)
for k, p in PATHS.items():
    print(f'  {k:22s} exists={os.path.exists(p)}  {p}')

In [ ]:
WESAD_SOURCE = 'complete'
PATHS['WESAD'] = '/content/WESAD_extracted/WESAD'
OUT_DIR = '/content/drive/MyDrive/bd_methods_paper/run_complete'
os.makedirs(OUT_DIR, exist_ok=True)
print('OUT_DIR:', OUT_DIR)
print('WESAD :', PATHS['WESAD'], os.path.exists(PATHS['WESAD']))

## Part A — Exact reported arrays

**A1** re-scores the saved models to confirm these are the arrays behind 75.6 / 82.3 / 82.6.
**A2** is the key test. The original Check A shuffled physio *across all rows*. Because rows were paired only by label,
a *within-class* shuffle gives a dataset with the same distribution as the "aligned" one.
If within-class F1 ≈ aligned F1, the pairing carries no cross-modal information, and the "alignment" claim is vacuous.
**A3** splits the 34-D physio vector into its sources (WESAD 26 · OBF actigraphy 4 · OBF clinical scores 4).
This assumes the original column order, which is checked.
**A4** checks for duplicate rows across the train/test split.

In [ ]:
#@title A · Exact-array diagnostics
npz_path = f"{PATHS['ORIGINAL_RESULTS_DIR']}/validation_data.npz"
A = {}
if not os.path.exists(npz_path):
    print(f'Not found: {npz_path}\nSkipping Part A. Part B rebuilds the arrays from raw data.')
else:
    d = np.load(npz_path)
    T, P, y = d['text_aligned'], d['physio_aligned'], d['labels_aligned'].astype(int)
    X = np.hstack([T, P])
    A['shapes'] = {'text': T.shape, 'physio': P.shape, 'n': len(y), 'class_counts': np.bincount(y).tolist()}
    print('Shapes:', A['shapes'])

    # A1: re-score the saved models on the saved test sets
    try:
        rep = {}
        for name, mfile, xk, yk in [('text', 'text_model.pkl', 'X_test_text', 'y_test_text'),
                                    ('physio', 'physio_model.pkl', 'X_test_physio', 'y_test_physio'),
                                    ('fusion', 'fusion_model.pkl', 'X_test_fusion', 'y_test_fusion')]:
            with open(f"{PATHS['ORIGINAL_RESULTS_DIR']}/{mfile}", 'rb') as f:
                m = pickle.load(f)
            rep[name] = macro_f1(d[yk].astype(int), m.predict(d[xk]))
        A['A1_saved_models_f1'] = rep
        print('A1 saved-model F1 (reported 0.756 / 0.823 / 0.826):', {k: round(v, 3) for k, v in rep.items()})
    except Exception as e:
        print('A1 skipped:', e)

    # Does refitting on the aligned arrays with the original split reproduce the fusion number?
    A['A1_refit_fusion_split_f1'] = split_f1(X, y)
    print('A1 refit fusion F1 on original split:', round(A['A1_refit_fusion_split_f1'], 3))

    # A2: shuffle controls, single original split (same protocol as Check A)
    rng = np.random.default_rng(SEED)
    aligned = [split_f1(X, y, SEED + i) for i in range(N_SHUFFLES)]
    across = [split_f1(np.hstack([T, P[perm_index(len(y), rng)]]), y, SEED + i) for i in range(N_SHUFFLES)]
    within = [split_f1(np.hstack([T, P[perm_index(len(y), rng, y)]]), y, SEED + i) for i in range(N_SHUFFLES)]
    A['A2_split'] = {'aligned': (np.mean(aligned), np.std(aligned)),
                     'across_class_shuffle': (np.mean(across), np.std(across)),
                     'within_class_shuffle': (np.mean(within), np.std(within))}
    # same comparison under repeated 5-fold CV (less split noise)
    rng = np.random.default_rng(SEED + 1)
    cv_al = cv_f1(X, y)
    cv_ac = [cv_f1(np.hstack([T, P[perm_index(len(y), rng)]]), y, n_repeats=1, seed=SEED + i)[0] for i in range(N_SHUFFLES)]
    cv_wi = [cv_f1(np.hstack([T, P[perm_index(len(y), rng, y)]]), y, n_repeats=1, seed=SEED + i)[0] for i in range(N_SHUFFLES)]
    A['A2_cv'] = {'aligned': cv_al, 'across_class_shuffle': (np.mean(cv_ac), np.std(cv_ac)),
                  'within_class_shuffle': (np.mean(cv_wi), np.std(cv_wi))}
    print('\nA2 shuffle controls (macro-F1, mean ± sd)')
    print(f"{'':24s}{'original split':>20s}{'repeated 5-fold CV':>22s}")
    for k in ['aligned', 'across_class_shuffle', 'within_class_shuffle']:
        print(f'{k:24s}{fmt(*A["A2_split"][k]):>20s}{fmt(*A["A2_cv"][k]):>22s}')

    # A3: feature-group ablation (assumes original column order: 26 WESAD, then OBF mean, sd, pctZeros, median, madrs, hads_d, asrs, mdq_pos)
    if P.shape[1] == 34:
        G = {'text_only': T, 'wesad_only': P[:, :26], 'obf_actigraphy_only': P[:, 26:30],
             'obf_clinical_scores_only': P[:, 30:34], 'physio_all': P,
             'physio_without_clinical_scores': P[:, :30],
             'fusion_all': X, 'fusion_without_clinical_scores': np.hstack([T, P[:, :30]])}
        A['A3_feature_groups'] = {}
        print('\nA3 feature-group ablation (macro-F1)')
        print(f"{'':34s}{'original split':>16s}{'repeated 5-fold CV':>22s}")
        for k, M in G.items():
            r = {'split': split_f1(M, y), 'cv': cv_f1(M, y)}
            A['A3_feature_groups'][k] = r
            print(f"{k:34s}{r['split']:>16.3f}{fmt(*r['cv']):>22s}")
    else:
        print(f'\nA3 skipped: physio has {P.shape[1]} columns, expected 34.')

    # A4: exact duplicates across the original split
    tr, te = original_split(y)
    def dup_frac(M):
        train_rows = {r.tobytes() for r in np.round(M[tr], 6)}
        return float(np.mean([r.tobytes() in train_rows for r in np.round(M[te], 6)]))
    A['A4_test_rows_with_exact_duplicate_in_train'] = {'text': dup_frac(T), 'physio': dup_frac(P),
                                                       'wesad_block': dup_frac(P[:, :26]) if P.shape[1] == 34 else None}
    print('\nA4 fraction of test rows with an exact duplicate in train:', A['A4_test_rows_with_exact_duplicate_in_train'])
RESULTS['A'] = A
save_results()

## Part B — Rebuild from raw data, with provenance
### B0 · Inventory

In [ ]:
#@title B0 · Dataset inventory
for k in ['WESAD', 'OBF']:
    p = PATHS[k]
    if os.path.isdir(p):
        entries = os.listdir(p)
        print(f'{k}: {len(entries)} entries, e.g. {sorted(entries)[:12]}')
    else:
        print(f'{k}: MISSING ({p})')
for k in ['KAGGLE_BD_CSV', 'MULTICLASS_CSV']:
    p = PATHS[k]
    print(f'{k}: ' + (f'{os.path.getsize(p)/1e6:.1f} MB' if os.path.exists(p) else f'MISSING ({p})'))

### B1 · WESAD: label-code audit and feature extraction
WESAD codebook, from the dataset's readme (**please confirm against `wesad_readme.pdf` in your copy**):
0 = transient/undefined, 1 = baseline, 2 = stress, 3 = amusement, 4 = meditation, 5–7 = ignore.
The original `map_to_bipolar_state()` maps codes {0,3}→Euthymia, {1,6}→Depression, {2,4}→Mania.
The audit below shows which real conditions end up in each "BD state".

In [ ]:
#@title B1  WESAD
WESAD_CODEBOOK = {0: 'transient', 1: 'baseline', 2: 'stress', 3: 'amusement', 4: 'meditation',
                  5: 'ignore_5', 6: 'ignore_6', 7: 'ignore_7'}
ORIGINAL_WESAD_MAP = {0: 0, 3: 0, 1: 1, 6: 1, 2: 2, 4: 2}   # exactly what map_to_bipolar_state() does

def quick_wesad_dataframe(subject_data, window_sec=1):
    """Verbatim feature extraction from multi_source_fusion.ipynb (26 features, 1-s windows)."""
    fs_chest, fs_wrist_bvp, fs_wrist_eda, fs_wrist_acc = 700, 64, 4, 32
    chest = subject_data['signal']['chest']; wrist = subject_data['signal']['wrist']; labels = subject_data['label']
    n_windows = len(chest['ECG']) // (fs_chest * window_sec)
    features = {}
    for s in ['ECG', 'EDA', 'EMG', 'Temp', 'Resp']:
        if s in chest:
            sig = chest[s][:n_windows * fs_chest].reshape((n_windows, fs_chest * window_sec))
            features[f'chest_{s.lower()}_mean'] = sig.mean(1); features[f'chest_{s.lower()}_std'] = sig.std(1)
    if 'ACC' in chest:
        acc = chest['ACC'][:n_windows * fs_chest].reshape((n_windows, fs_chest * window_sec, 3))
        for i, a in enumerate('xyz'):
            features[f'chest_acc_{a}_mean'] = acc[:, :, i].mean(1); features[f'chest_acc_{a}_std'] = acc[:, :, i].std(1)
    if 'BVP' in wrist:
        bvp = wrist['BVP'][:n_windows * fs_wrist_bvp * window_sec].reshape((n_windows, fs_wrist_bvp * window_sec))
        features['wrist_bvp_mean'] = bvp.mean(1); features['wrist_bvp_std'] = bvp.std(1)
    if 'EDA' in wrist:
        eda = wrist['EDA'][:n_windows * fs_wrist_eda * window_sec].reshape((n_windows, fs_wrist_eda * window_sec))
        features['wrist_eda_mean'] = eda.mean(1); features['wrist_eda_std'] = eda.std(1)
    if 'ACC' in wrist:
        acc = wrist['ACC'][:n_windows * fs_wrist_acc * window_sec].reshape((n_windows, fs_wrist_acc * window_sec, 3))
        for i, a in enumerate('xyz'):
            features[f'wrist_acc_{a}_mean'] = acc[:, :, i].mean(1); features[f'wrist_acc_{a}_std'] = acc[:, :, i].std(1)
    ls = labels[:n_windows * fs_chest].reshape((n_windows, fs_chest * window_sec))
    features['label'] = np.array([np.bincount(w.astype(int)).argmax() for w in ls])
    return pd.DataFrame(features)

wesad_cache = f'{CACHE_DIR}/wesad_windows_{WESAD_SOURCE}.pkl'
if os.path.exists(wesad_cache):
    wesad_all, wesad_seconds = pd.read_pickle(wesad_cache)
    print('Loaded cached WESAD windows:', wesad_all.shape)
else:
    frames, sec_rows = [], []
    for sid in range(2, 18):    # S2–S17, same loop as the original
        path = f"{PATHS['WESAD']}/S{sid}/S{sid}.pkl"
        try:
            with open(path, 'rb') as f:
                data = pickle.load(f, encoding='latin1')
            lab = np.asarray(data['label']).astype(int)
            sec_rows.append({'subject': f'S{sid}', **{WESAD_CODEBOOK[c]: round(float((lab == c).sum()) / 700, 1)
                                                      for c in range(8)}})
            df = quick_wesad_dataframe(data)
            df['subject'] = f'S{sid}'; df['window'] = np.arange(len(df))
            frames.append(df); print(f'  S{sid}: {len(df)} windows')
            del data
        except Exception as e:
            print(f'  S{sid}: skipped ({e})')

    if len(frames) == 0:
        print('\n[WARNING] CONFIG NEEDED: No WESAD datasets could be found or loaded. Please check your PATHS config in cell 0.')
        wesad_all = pd.DataFrame(columns=['label', 'subject', 'window', 'condition', 'bipolar_state'])
        wesad_seconds = pd.DataFrame()
    else:
        wesad_all = pd.concat(frames, ignore_index=True)
        wesad_seconds = pd.DataFrame(sec_rows)
        pd.to_pickle((wesad_all, wesad_seconds), wesad_cache)

if len(wesad_all) > 0:
    WESAD_FEATURES = [c for c in wesad_all.columns if c not in ('label', 'subject', 'window')]
    assert len(WESAD_FEATURES) == 26, f'Expected 26 WESAD features, got {len(WESAD_FEATURES)}'

    print('\nSeconds of each raw WESAD code per subject:')
    print(wesad_seconds.to_string(index=False))
    wesad_all['condition'] = wesad_all['label'].map(WESAD_CODEBOOK)
    wesad_all['bipolar_state'] = wesad_all['label'].map(ORIGINAL_WESAD_MAP)
    xt = pd.crosstab(wesad_all['condition'], wesad_all['bipolar_state'].map(dict(enumerate(CLASS_NAMES))).fillna('dropped'))
    print('\nWindows: WESAD condition (codebook)  assigned "BD state" (original mapping):')
    print(xt)
    RESULTS['B1_wesad'] = {'n_subjects': wesad_all['subject'].nunique(), 'n_windows': len(wesad_all),
                           'condition_x_state': xt.to_dict()}
    save_results()
else:
    print('\nSkipped calculations: No WESAD data loaded.')

### B2 · OBF-Psychiatric: replicate the cluster labels, then test circularity
The original pipeline concatenated every CSV in the OBF folder, imputed missing values with column means,
and labeled rows by GMM clusters of autoencoder latents of 8 features, **4 of which are clinical rating scales**.
Those same 8 features were then used as model inputs. The checks below:
(i) inspect what the concatenation actually produced (including rows that were *entirely* imputed);
(ii) test whether the cluster labels are recoverable from the clinical scores alone;
(iii) compare cluster labels against the real diagnoses already in the dataset.

In [ ]:
#@title B2 · OBF-Psychiatric
OBF_FEATURES = ['mean', 'sd', 'pctZeros', 'median', 'madrs', 'hads_d', 'asrs', 'mdq_pos']
OBF_ACTIGRAPHY, OBF_CLINICAL = OBF_FEATURES[:4], OBF_FEATURES[4:]

# Paste a recorded order here to reproduce an earlier run exactly; None = use this filesystem's.
OBF_FILE_ORDER_REPLAY = None

frames = []
OBF_FILE_ORDER = []
_listing = OBF_FILE_ORDER_REPLAY if OBF_FILE_ORDER_REPLAY else os.listdir(PATHS['OBF'])
for fn in _listing:
    fp = os.path.join(PATHS['OBF'], fn)
    if os.path.isfile(fp) and fn.endswith('.csv'):
        try:
            df = pd.read_csv(fp)
            df['__source_file'] = fn
            frames.append(df)
            OBF_FILE_ORDER.append(fn)
        except Exception as e:
            print('Error reading', fn, e)
obf_raw = pd.concat(frames, ignore_index=True)
print(f'OBF: {len(frames)} CSV files, {len(obf_raw)} rows')
for fn, g in obf_raw.groupby('__source_file'):
    cols = [c for c in g.columns if g[c].notna().any() and c != '__source_file']
    print(f'  {fn}: {len(g)} rows | columns: {cols[:15]}{" ..." if len(cols) > 15 else ""}')

missing = [c for c in OBF_FEATURES if c not in obf_raw.columns]
if missing:
    raise ValueError(f'CONFIG NEEDED: OBF columns {missing} not found. Columns are: {list(obf_raw.columns)}')

def _detect(cands):
    low = {c.lower(): c for c in obf_raw.columns}
    return next((low[c] for c in cands if c in low), None)
OBF_ID_COL = OBF_ID_COL or _detect(['number', 'id', 'user', 'userid', 'patient', 'subject', 'participant'])
OBF_DIAG_COL = OBF_DIAG_COL or _detect(['diagnosis', 'diag', 'group', 'class', 'label', 'condition', 'category'])
print(f'\nOBF_ID_COL = {OBF_ID_COL!r}   OBF_DIAG_COL = {OBF_DIAG_COL!r}  (set in cell 0 if wrong)')

def _col_or_missing(col, unique_missing=False):
    """Column as strings; missing values (e.g. rows from CSVs with a different schema) made explicit."""
    v = obf_raw[col].astype(object)
    miss = v.isna().values
    out = v.where(~v.isna(), '<missing>').astype(str).values
    if unique_missing:   # each ID-less row becomes its own group so grouped CV stays valid
        out = np.where(miss, np.array([f'<missing_{i}>' for i in range(len(v))]), out)
    return out, miss
if OBF_ID_COL:
    obf_person, id_missing = _col_or_missing(OBF_ID_COL, unique_missing=True)
    print(f'Rows with no {OBF_ID_COL!r} value: {int(id_missing.sum())}')
else:
    obf_person = np.array([f'row_{i}' for i in range(len(obf_raw))])

all_nan = obf_raw[OBF_FEATURES].isna().all(axis=1)
some_nan = obf_raw[OBF_FEATURES].isna().any(axis=1)
print(f'Rows with ALL 8 features missing (fully imputed with column means): {int(all_nan.sum())} / {len(obf_raw)}')
print(f'Rows with ANY feature missing: {int(some_nan.sum())}')
print('Missing rate per feature:', obf_raw[OBF_FEATURES].isna().mean().round(3).to_dict())

# ---- Replicate the original labeling (cells 7–10 of multi_source_fusion.ipynb) ----
X_obf = obf_raw[OBF_FEATURES].fillna(obf_raw[OBF_FEATURES].mean())
obf_tr, obf_te = train_test_split(np.arange(len(X_obf)), test_size=0.2, random_state=42)
scaler = StandardScaler().fit(X_obf.values[obf_tr])
obf_scaled = scaler.transform(X_obf.values)          # same values as the original's per-split transform

if SMOKE_TEST:   # torch unavailable in the test sandbox: PCA stand-in for the autoencoder
    from sklearn.decomposition import PCA
    pca = PCA(3, random_state=42).fit(obf_scaled[obf_tr]); Z = pca.transform(obf_scaled)
else:
    import torch, torch.nn as nn
    torch.manual_seed(42)   # the original did not seed torch, so cluster labels may differ slightly from the reported run
    class Autoencoder(nn.Module):
        def __init__(self, d=8, k=3):
            super().__init__()
            self.encoder = nn.Sequential(nn.Linear(d, 64), nn.ReLU(), nn.Linear(64, k), nn.ReLU())
            self.decoder = nn.Sequential(nn.Linear(k, 64), nn.ReLU(), nn.Linear(64, d))
        def forward(self, x): return self.decoder(self.encoder(x))
    ae = Autoencoder(obf_scaled.shape[1]); opt = torch.optim.Adam(ae.parameters(), lr=1e-3); lossf = nn.MSELoss()
    Xt = torch.FloatTensor(obf_scaled[obf_tr])
    for epoch in range(50):
        for i in range(0, len(Xt), 32):
            b = Xt[i:i + 32]; opt.zero_grad(); l = lossf(ae(b), b); l.backward(); opt.step()
    ae.eval()
    with torch.no_grad():
        Z = ae.encoder(torch.FloatTensor(obf_scaled)).numpy()

from sklearn.mixture import GaussianMixture
gmm = GaussianMixture(n_components=3, random_state=42).fit(Z[obf_tr])
obf_labels = gmm.predict(Z)                            # cluster_to_state was the identity {0:0, 1:1, 2:2}
print('\nGMM "BD state" counts:', np.bincount(obf_labels, minlength=3).tolist())
print('Mean raw clinical scores per cluster (this is how clusters were named):')
print(obf_raw.assign(cluster=obf_labels).groupby('cluster')[OBF_CLINICAL].mean().round(2))

# ---- Circularity: can each feature group recover the cluster labels? ----
B2 = {'n_rows': len(obf_raw), 'n_fully_imputed_rows': int(all_nan.sum()),
      'gmm_counts': np.bincount(obf_labels, minlength=3).tolist()}
yl = obf_labels.astype(int)
grp = obf_person if OBF_ID_COL else None
print('\nRecovering the GMM labels from feature subsets (macro-F1, repeated 5-fold CV):')
for name, cols in [('clinical_scores_only', [4, 5, 6, 7]), ('actigraphy_only', [0, 1, 2, 3]), ('all_8', list(range(8)))]:
    r = cv_f1(obf_scaled[:, cols], yl)
    rg = cv_f1(obf_scaled[:, cols], yl, groups=grp) if grp is not None else None
    B2[f'recover_gmm_from_{name}'] = {'row_cv': r, 'subject_grouped_cv': rg}
    print(f'  {name:22s} row-CV {fmt(*r)}' + (f' | subject-grouped {fmt(*rg)}' if rg else ''))

if OBF_DIAG_COL:
    diag, diag_missing = _col_or_missing(OBF_DIAG_COL)
    ct = pd.crosstab(diag, obf_labels)
    print(f'\nReal diagnosis ({OBF_DIAG_COL}) × GMM "BD state" (0=Euthymia, 1=Depression, 2=Mania):')
    print(ct)
    B2['diagnosis_x_gmm'] = ct.to_dict(); B2['ARI_diagnosis_vs_gmm'] = adjusted_rand_score(diag, obf_labels)
    print('ARI(diagnosis, GMM labels):', round(B2['ARI_diagnosis_vs_gmm'], 3))
    if grp is not None:   # the honest version of the task this dataset supports
        keep = ~diag_missing
        dy = pd.factorize(diag[keep])[0]
        if len(np.unique(dy)) > 1 and np.bincount(dy).min() >= 2:
            B2['diagnosis_from_actigraphy_subject_grouped'] = cv_f1(obf_scaled[keep][:, :4], dy, groups=grp[keep])
            print('Real diagnosis from actigraphy only, subject-grouped CV:',
                  fmt(*B2['diagnosis_from_actigraphy_subject_grouped']))
else:
    print('\nNo diagnosis column detected; set OBF_DIAG_COL in cell 0 to run the diagnosis comparison.')
B2['source_file_order'] = OBF_FILE_ORDER
RESULTS['B2_obf'] = B2
save_results()

### B3 · Text: replicate labels, then test circularity and source confounding
Kaggle BD labels were KMeans clusters of the *same* BERT embeddings used as model input, named by VADER sentiment.
Multi-class labels were mapped by hand ("bipolar"→Depression, "psychotic"→Mania).
We also test whether the label is confounded with *which corpus* a post came from.

In [ ]:
#@title B3 · Text corpora
def clean_text(text):
    text = re.sub(r'http\S+', '', str(text)); text = re.sub(r'[^a-zA-Z\s]', '', text); return text.lower()

kag = pd.read_csv(PATHS['KAGGLE_BD_CSV'], encoding='latin1', delimiter=';')
kag['post'] = kag['post'].apply(clean_text)
mc = pd.read_csv(PATHS['MULTICLASS_CSV'])
low = {c.lower(): c for c in mc.columns}
text_col = next(low[c] for c in ['tweet', 'tweets', 'text'] if c in low)
label_col = next(low[c] for c in ['label', 'labels', 'class'] if c in low)
mc[text_col] = mc[text_col].apply(clean_text)
LABEL_MAP = {'major depressive': 1, 'atypical': 1, 'psychotic': 2, 'no': 0, 'bipolar': 1, 'postpartum': -1}
print('Multi-class raw label values:', mc[label_col].value_counts().to_dict())
mc['bd_label'] = mc[label_col].map(LABEL_MAP)
unmapped = mc['bd_label'].isna().sum()
mc = mc[mc['bd_label'] != -1].dropna(subset=['bd_label'])
print(f'Rows silently dropped as unmapped: {unmapped}; kept: {len(mc)}')

def embed(texts, cache_name):
    path = f'{CACHE_DIR}/{cache_name}.npy'
    if os.path.exists(path):
        E = np.load(path)
        if len(E) == len(texts): return E
    if SMOKE_TEST:
        E = np.random.default_rng(0).normal(size=(len(texts), 768)).astype(np.float32)
    else:
        import torch
        from transformers import BertTokenizer, BertModel
        dev = 'cuda' if torch.cuda.is_available() else 'cpu'
        tok = BertTokenizer.from_pretrained('bert-base-uncased'); bert = BertModel.from_pretrained('bert-base-uncased').to(dev).eval()
        out = []
        for i in range(0, len(texts), BERT_BATCH):
            enc = tok(list(texts[i:i + BERT_BATCH]), return_tensors='pt', truncation=True, max_length=512, padding=True).to(dev)
            with torch.no_grad():
                h = bert(**enc).last_hidden_state
            m = enc['attention_mask'].unsqueeze(-1).float()
            out.append(((h * m).sum(1) / m.sum(1)).cpu().numpy())   # masked mean == original unpadded mean
            if (i // BERT_BATCH) % 50 == 0: print(f'  {cache_name}: {i}/{len(texts)}')
        E = np.vstack(out)
    np.save(path, E); return E

E_kag = embed(kag['post'].values, 'emb_kaggle_bd')
E_mc = embed(mc[text_col].values, 'emb_multiclass')

from sklearn.cluster import KMeans
km = KMeans(n_clusters=3, random_state=42).fit(E_kag)   # NB: sklearn's n_init default changed in 1.4; cluster IDs may differ from the original run
kag_labels = km.labels_                                  # cluster_to_label was the identity
B3 = {'kaggle_cluster_counts': np.bincount(kag_labels, minlength=3).tolist()}
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    va = SentimentIntensityAnalyzer()
    sent = np.array([va.polarity_scores(t)['compound'] for t in kag['post'].values])
    B3['vader_mean_by_cluster'] = {int(c): float(sent[kag_labels == c].mean()) for c in range(3)}
    print('Mean VADER sentiment per KMeans cluster (basis for naming):', B3['vader_mean_by_cluster'])
except ImportError:
    print('(pip install vaderSentiment to reproduce the sentiment-based naming step)')

text_emb = np.vstack([E_kag, E_mc]).astype(np.float32)
text_labels = np.concatenate([kag_labels, mc['bd_label'].astype(int).values])
text_source = np.array(['kaggle_bd_reddit'] * len(E_kag) + ['multiclass_twitter'] * len(E_mc))

# Circularity: the Kaggle labels are a deterministic function of the input features
B3['kaggle_labels_recovered_by_nearest_centroid'] = float((km.predict(E_kag) == kag_labels).mean())
print('Kaggle labels recovered exactly by nearest KMeans centroid:', B3['kaggle_labels_recovered_by_nearest_centroid'])

# Source confounding
ct = pd.crosstab(text_source, pd.Series(text_labels).map(dict(enumerate(CLASS_NAMES))).values)
print('\nCorpus × label:'); print(ct)
B3['source_x_label'] = ct.to_dict()
rng = np.random.default_rng(SEED)
sub = np.concatenate([rng.choice(np.where(text_source == s)[0], min(2000, (text_source == s).sum()), replace=False)
                      for s in np.unique(text_source)])
src_y = pd.factorize(text_source[sub])[0]
B3['source_identity_probe_cv'] = cv_f1(text_emb[sub], src_y, n_repeats=1)
print('Corpus-identity probe (predict Reddit vs Twitter from embeddings), macro-F1:', fmt(*B3['source_identity_probe_cv']))
maj = pd.Series(text_labels).groupby(text_source).agg(lambda s: s.value_counts().idxmax())
B3['label_from_source_only_f1'] = macro_f1(text_labels, maj.loc[text_source].values)
print('Macro-F1 from predicting each corpus\'s majority label (no text used):', round(B3['label_from_source_only_f1'], 3))
RESULTS['B3_text'] = B3
save_results()

### B4 · Rebuild the fused dataset with provenance, and replicate the reported numbers
Same steps and seeds as the original. Each fused row also carries its WESAD subject/window, its OBF row/person,
and its text corpus. `build_fused(seed)` lets us redraw the label-conditional pairing with other seeds.

In [ ]:
#@title B4 · Rebuild & replicate
from sklearn.utils import resample

wc = wesad_all.dropna(subset=['bipolar_state']).reset_index(drop=True)
wc['bipolar_state'] = wc['bipolar_state'].astype(int); wc['row_id'] = np.arange(len(wc))

if len(wc) == 0:
    print('[WARNING] Cannot balance WESAD because wesad_all is empty. Creating an empty placeholder.')
    wesad_balanced = pd.DataFrame(columns=list(wesad_all.columns) + ['row_id'])
    dup_rate = 0.0
else:
    min_count = wc['bipolar_state'].value_counts().min()
    wesad_balanced = pd.concat([resample(wc[wc['bipolar_state'] == c], n_samples=min_count, random_state=42)   # replace=True (sklearn default)
                                for c in [0, 1, 2]]).sample(frac=1, random_state=42).reset_index(drop=True)
    dup_rate = 1 - wesad_balanced['row_id'].nunique() / len(wesad_balanced)
    print(f'WESAD balanced: {len(wesad_balanced)} rows; {dup_rate:.1%} are duplicates created by resampling with replacement')

obf_df = pd.DataFrame(obf_scaled, columns=[f'obf_feat_{i}' for i in range(8)])
obf_df['bipolar_state'] = obf_labels
obf_df['obf_person'] = obf_person

def build_fused(seed=42):
    """Cells 11 and 17 of the original, with metadata carried along. seed=42 reproduces the original draw."""
    if len(wesad_balanced) == 0:
        # Return empty dictionary or basic structure to prevent cascading crashes
        return {
            'T': np.empty((0, 768)), 'P': np.empty((0, 34)), 'y': np.array([], dtype=int),
            'meta': pd.DataFrame(columns=['label', 'text_source', 'wesad_subject', 'wesad_window', 'wesad_row_id', 'obf_row', 'obf_person']),
            'physio_full': np.empty((0, 34)), 'physio_labels': np.array([])
        }
    np.random.seed(seed)
    wl = {c: wesad_balanced.index[wesad_balanced['bipolar_state'] == c].values for c in range(3)}
    ol = {c: obf_df.index[obf_df['bipolar_state'] == c].values for c in range(3)}
    mpc = min(min(len(wl[c]) for c in range(3)), min(len(ol[c]) for c in range(3)))
    wi, oi, lab = [], [], []
    for c in range(3):
        wi.extend(np.random.choice(wl[c], mpc, replace=False)); oi.extend(np.random.choice(ol[c], mpc, replace=False)); lab.extend([c] * mpc)
    wi, oi, lab = map(np.array, (wi, oi, lab)); s = np.random.permutation(len(wi)); wi, oi, lab = wi[s], oi[s], lab[s]
    physio = np.hstack([wesad_balanced.loc[wi, WESAD_FEATURES].values, obf_df.loc[oi, [f'obf_feat_{i}' for i in range(8)]].values])
    # cell 17
    np.random.seed(seed)
    ti, pi, l2 = [], [], []
    for c in range(3):
        t_idx = np.where(text_labels == c)[0]; p_idx = np.where(lab == c)[0]; n = min(len(t_idx), len(p_idx))
        ti.extend(np.random.choice(t_idx, n, replace=False)); pi.extend(np.random.choice(p_idx, n, replace=False)); l2.extend([c] * n)
    ti, pi, l2 = map(np.array, (ti, pi, l2)); s = np.random.permutation(len(ti)); ti, pi, l2 = ti[s], pi[s], l2[s]
    meta = pd.DataFrame({'label': l2, 'text_source': text_source[ti],
                         'wesad_subject': wesad_balanced.loc[wi[pi], 'subject'].values,
                         'wesad_window': wesad_balanced.loc[wi[pi], 'window'].values,
                         'wesad_row_id': wesad_balanced.loc[wi[pi], 'row_id'].values,
                         'obf_row': oi[pi], 'obf_person': obf_df.loc[oi[pi], 'obf_person'].values})
    return {'T': text_emb[ti], 'P': physio[pi], 'y': l2.astype(int), 'meta': meta,
            'physio_full': physio, 'physio_labels': lab}

F = build_fused(42)
print('Fused rows:', len(F['y']), '| per class:', np.bincount(F['y']).tolist() if len(F['y']) > 0 else [],
      '| WESAD subjects:', F['meta']['wesad_subject'].nunique(), '| corpora:', F['meta']['text_source'].value_counts().to_dict())

tr_t, te_t = train_test_split(np.arange(len(text_labels)), test_size=0.2, stratify=text_labels, random_state=42)
if len(F['y']) > 0:
    rep = {'text_only_full_corpus': macro_f1(text_labels[te_t], rf().fit(text_emb[tr_t], text_labels[tr_t]).predict(text_emb[te_t])),
           'physio_only': split_f1(F['physio_full'], F['physio_labels']),
           'fusion': split_f1(np.hstack([F['T'], F['P']]), F['y'])}
    print('Replication on original splits (reported 0.756 / 0.823 / 0.826):', {k: round(v, 3) for k, v in rep.items()})
else:
    rep = {'text_only_full_corpus': 0.0, 'physio_only': 0.0, 'fusion': 0.0}
    print('Replication skipped because no fused rows are available.')

RESULTS['B4_replication'] = {'wesad_duplicate_rate': dup_rate, 'n_fused': len(F['y']), **rep}
save_results()

### B5 · Leakage audit
How "unseen" is the test set? For each fused test row, do we find the same WESAD subject, the same window,
a window less than 5 s away, or the same OBF person in the training set? Then compare a random-window split with
leave-subject-out on WESAD alone, using a larger sample.

In [ ]:
#@title B5 · Leakage
m = F['meta']; tr, te = original_split(F['y'])
def frac(cond): return float(np.mean(cond))
tr_m, te_m = m.iloc[tr], m.iloc[te]
near = []
for _, r in te_m.iterrows():
    same = tr_m[tr_m['wesad_subject'] == r['wesad_subject']]
    near.append(len(same) > 0 and (np.abs(same['wesad_window'] - r['wesad_window']) < 5).any())
B5 = {'test_rows_same_wesad_subject_in_train': frac(te_m['wesad_subject'].isin(tr_m['wesad_subject'])),
      'test_rows_same_wesad_window_in_train': frac(te_m['wesad_row_id'].isin(tr_m['wesad_row_id'])),
      'test_rows_wesad_window_within_5s_in_train': frac(near),
      'test_rows_same_obf_person_in_train': frac(te_m['obf_person'].isin(tr_m['obf_person'])) if OBF_ID_COL else None}
for k, v in B5.items(): print(f'{k:45s} {v}')

rng = np.random.default_rng(SEED)
samp = np.concatenate([rng.choice(np.where(wc['bipolar_state'] == c)[0], min(2000, (wc['bipolar_state'] == c).sum()), replace=False) for c in range(3)])
Xw, yw, gw = wc.loc[samp, WESAD_FEATURES].values, wc.loc[samp, 'bipolar_state'].values, wc.loc[samp, 'subject'].values
B5['wesad_random_window_cv'] = cv_f1(Xw, yw, n_repeats=2)
loso = np.empty_like(yw)
for trn, tst in GroupKFold(len(np.unique(gw))).split(Xw, yw, gw):
    loso[tst] = rf().fit(Xw[trn], yw[trn]).predict(Xw[tst])
B5['wesad_leave_one_subject_out'] = macro_f1(yw, loso)
print(f"\nWESAD-only, mapped 3 classes: random-window CV {fmt(*B5['wesad_random_window_cv'])} vs leave-one-subject-out {B5['wesad_leave_one_subject_out']:.3f}")
RESULTS['B5_leakage'] = B5
save_results()

### B6 · Waterfall
Cumulative corrections, each evaluated over `N_BUILDS` independent draws of the pairing.
At every stage we also report the within-class and across-class shuffle diagnostics and the unimodal baselines.

In [ ]:
#@title B6 · Waterfall
CLIN = list(range(30, 34))   # OBF clinical-score columns inside P
def stage_eval(T, P, y, groups, rng, reps):
    X = np.hstack([T, P])
    wi = perm_index(len(y), rng, y); ac = perm_index(len(y), rng)
    g_w = None if groups is None else groups[wi]; g_a = None if groups is None else groups[ac]
    # groups follow the physio rows they belong to
    return {'fusion': cv_f1(X, y, groups, reps)[0], 'text_only': cv_f1(T, y, groups, reps)[0],
            'physio_only': cv_f1(P, y, groups, reps)[0],
            'fusion_within_class_shuffle': cv_f1(np.hstack([T, P[wi]]), y, g_w, reps)[0],
            'fusion_across_class_shuffle': cv_f1(np.hstack([T, P[ac]]), y, g_a, reps)[0]}

stages = {'S1 repeated CV (same features)': dict(drop_clin=False, group=None),
          'S2 + clinical scores removed': dict(drop_clin=True, group=None),
          'S3 + grouped by WESAD subject': dict(drop_clin=True, group='wesad_subject')}
if OBF_ID_COL:
    stages['S3b + grouped by OBF person (alt.)'] = dict(drop_clin=True, group='obf_person')

rows = {k: [] for k in stages}
for b in range(N_BUILDS):
    Fb = build_fused(42 + b); rng = np.random.default_rng(1000 + b)
    for name, cfg in stages.items():
        P = np.delete(Fb['P'], CLIN, axis=1) if cfg['drop_clin'] else Fb['P']
        g = Fb['meta'][cfg['group']].values if cfg['group'] else None
        rows[name].append(stage_eval(Fb['T'], P, Fb['y'], g, rng, WATERFALL_CV_REPEATS))
    print(f'  build {b + 1}/{N_BUILDS} done')

W = {'S0 reported protocol (single split, build 42)': {'fusion': rep['fusion']}}
for name, lst in rows.items():
    W[name] = {k: (float(np.mean([r[k] for r in lst])), float(np.std([r[k] for r in lst]))) for k in lst[0]}
cols = ['fusion', 'text_only', 'physio_only', 'fusion_within_class_shuffle', 'fusion_across_class_shuffle']
print(f"\n{'stage':46s}" + ''.join(f'{c[:22]:>24s}' for c in cols))
for name, r in W.items():
    print(f'{name:46s}' + ''.join(f"{(fmt(*r[c]) if isinstance(r.get(c), tuple) else (fmt(r[c]) if c in r else '-')):>24s}" for c in cols))
RESULTS['B6_waterfall'] = W
save_results()

### B7 · Calibration audit
(a) Replicates the original: isotonic calibration fitted on the **training** set of a prefit forest, ECE on 29 test rows,
at several bin counts, with bootstrap CIs. (b) Calibration fitted by internal CV and evaluated on held-out folds over 20 splits.

In [ ]:
#@title B7 · Calibration
X = np.hstack([F['T'], F['P']]); y = F['y']; tr, te = original_split(y)
base = rf().fit(X[tr], y[tr])
try:
    cal = CalibratedClassifierCV(base, method='isotonic', cv='prefit').fit(X[tr], y[tr])
except Exception:
    from sklearn.frozen import FrozenEstimator   # sklearn >= 1.6
    cal = CalibratedClassifierCV(FrozenEstimator(base), method='isotonic').fit(X[tr], y[tr])
B7 = {'n_test': len(te), 'original_protocol': {}}
rng = np.random.default_rng(SEED)
for name, model in [('uncalibrated', base), ('isotonic_fit_on_train', cal)]:
    pr = model.predict_proba(X[te])
    for nb in [5, 10, 15]:
        boots = [ece(y[te][i], pr[i], nb) for i in (rng.integers(0, len(te), len(te)) for _ in range(1000))]
        B7['original_protocol'][f'{name}_bins{nb}'] = {'ece': ece(y[te], pr, nb), 'ci95': np.percentile(boots, [2.5, 97.5]).tolist()}
for k, v in B7['original_protocol'].items():
    print(f"{k:32s} ECE {v['ece']:.3f}  95% CI [{v['ci95'][0]:.3f}, {v['ci95'][1]:.3f}]")
heldout = []
for i, (trn, tst) in enumerate(StratifiedShuffleSplit(20, test_size=0.2, random_state=SEED).split(X, y)):
    c = CalibratedClassifierCV(rf(SEED + i), method='isotonic', cv=5).fit(X[trn], y[trn])
    heldout.append(ece(y[tst], c.predict_proba(X[tst]), 10))
B7['heldout_cv_calibrated_ece10'] = {'mean': float(np.mean(heldout)), 'p2.5': float(np.percentile(heldout, 2.5)),
                                     'p97.5': float(np.percentile(heldout, 97.5))}
print('Held-out calibration over 20 splits (10 bins):', B7['heldout_cv_calibrated_ece10'])
RESULTS['B7_calibration'] = B7
save_results()

### B8 · OBF source-file audit

Where the cluster labels came from. The pipeline concatenated every CSV in the OBF folder,
so each row carries only its own file's columns; `fillna(mean)` filled the rest. These cells
show that the resulting missingness pattern — not the measurements — is what the mixture
model separated.


In [ ]:
#@title B8 · Source-file audit
# Which source file did each cluster come from, and what does each file actually contain?
print('source file x GMM cluster'); print(pd.crosstab(obf_raw['__source_file'], obf_labels))
print('\nfeature coverage by source file (fraction non-missing)')
print(obf_raw.groupby('__source_file')[OBF_FEATURES].apply(lambda g: g.notna().mean()).round(2))

# Fully-imputed rows: do they explain any one cluster?
print('\nGMM cluster x all-8-features-missing'); print(pd.crosstab(obf_labels, all_nan.values))

# The frozen-half signature inside the fused data.
B8 = {}
for name, sl in [('actigraphy', slice(26, 30)), ('clinical', slice(30, 34))]:
    for c in range(3):
        rows_ = np.round(F['P'][F['y'] == c, sl], 6)
        u = len({r.tobytes() for r in rows_})
        B8[f'{name}_class{c}_unique'] = u
        print(f'{name:11s} class {c}: {len(rows_)} rows, {u} unique')

# Can the label be read off the missingness mask alone?
mask = obf_raw[OBF_FEATURES].isna().values.astype(int)
B8['label_from_missingness_mask'] = cv_f1(mask, obf_labels.astype(int))
print('\nGMM label from missingness mask only (floor 0.333):', fmt(*B8['label_from_missingness_mask']))
RESULTS['B8_source_audit'] = B8; save_results()

### B9 · The join that was never performed

OBF ships a derived feature table, per-group metadata tables, and raw per-participant
activity series. `features.csv` and the info tables share a participant key, so they were
meant to be merged. They were concatenated instead.


In [ ]:
#@title B9 · The missing join
feat = pd.read_csv(f"{PATHS['OBF']}/features.csv")
info_raw = pd.concat([pd.read_csv(f"{PATHS['OBF']}/{g}-info.csv").assign(group=g)
                      for g in ['adhd','clinical','depression','control','schizophrenia']],
                     ignore_index=True)

B9 = {'features_people': int(feat['user'].nunique()), 'features_rows': len(feat),
      'info_people': int(info_raw['number'].nunique()),
      'key_overlap': len(set(feat['user'].astype(str)) & set(info_raw['number'].astype(str)))}
print(B9)
print('\nfeatures.csv own class column:', feat['class'].value_counts().to_dict())

# Only 77 of 162 people appear in features.csv - exactly the three groups with no clinical scales.
print('\npeople in features.csv, by info group:')
print(info_raw[info_raw['number'].astype(str).isin(feat['user'].astype(str))]
      .groupby('group')['number'].nunique().to_dict())
print('people with clinical scales but no features.csv row:')
print(info_raw[~info_raw['number'].astype(str).isin(feat['user'].astype(str))]
      .groupby('group')['number'].nunique().to_dict())

# The raw series are present for all 162, in per-group subfolders.
for d in ['adhd','clinical','depression','control','schizophrenia']:
    fs = [f for f in os.listdir(f"{PATHS['OBF']}/{d}") if f.endswith('.csv')]
    print(f'  {d}: {len(fs)} participant files')
print('\nexample raw series:')
print(pd.read_csv(f"{PATHS['OBF']}/adhd/{sorted(os.listdir(PATHS['OBF']+'/adhd'))[0]}").head(3))
RESULTS['B9_join'] = B9; save_results()

## Section 6 — the corrected dataset

Built the way the data actually supports: person-day features computed from the raw series
for all 162 participants, joined to the metadata on the participant key, with the filter
stated explicitly.

The join key is the filename stem (`adhd_41`), which is exactly the value in `info['number']`.
Parsing an integer out of the filename and merging on that is what silently produced an
all-NaN join on the first attempt.


In [ ]:
#@title S6 · Build the corrected dataset
GROUPS = ['adhd', 'clinical', 'depression', 'control', 'schizophrenia']
S6 = {}

# ---- 6.1 person-day features from the raw series
rows_ = []
for g in GROUPS:
    for fn in sorted(os.listdir(f"{PATHS['OBF']}/{g}")):
        if not fn.endswith('.csv'):
            continue
        person = fn[:-4]                      # 'adhd_41' - matches info['number'] exactly
        df = pd.read_csv(f"{PATHS['OBF']}/{g}/{fn}")
        for day, d in df.groupby('date'):
            a = d['activity'].values.astype(float)
            rows_.append({'group': g, 'person': person, 'date': day,
                          'n_min': len(a), 'n_dup_ts': int(d['timestamp'].duplicated().sum()),
                          'mean': a.mean(), 'sd': a.std(), 'pctZeros': float((a == 0).mean()),
                          'median': float(np.median(a)), 'q25': float(np.percentile(a, 25)),
                          'q75': float(np.percentile(a, 75))})
pd_all = pd.DataFrame(rows_)

keep = (pd_all.n_min == 1440) & (pd_all.n_dup_ts == 0) & (pd_all['mean'] > 0)
pdf = pd_all[keep].reset_index(drop=True)
S6['filter'] = {'person_days_raw': len(pd_all), 'person_days_kept': len(pdf),
                'dropped_incomplete': int((pd_all.n_min != 1440).sum()),
                'dropped_dup_timestamps': int((pd_all.n_dup_ts > 0).sum()),
                'dropped_zero_activity': int((pd_all['mean'] == 0).sum())}
print(S6['filter'])
print(pdf.groupby('group').agg(people=('person', 'nunique'), days=('date', 'size')))

# ---- 6.2 join on the participant key
info = pd.concat([pd.read_csv(f"{PATHS['OBF']}/{g}-info.csv").assign(group=g) for g in GROUPS],
                 ignore_index=True)
info['person'] = info['number'].astype(str)
D = pdf.merge(info.drop(columns=['group', 'number']), on='person', how='left',
              validate='many_to_one')

# Validate on a column supplied ONLY by the right-hand table - never on the join key itself.
print('\nunmatched person-days:', int(D['age'].isna().sum()),
      '| people with metadata:', D.loc[D['age'].notna(), 'person'].nunique(),
      'of', pdf['person'].nunique())

ACT = ['mean', 'sd', 'pctZeros', 'median', 'q25', 'q75']
SCALES = [c for c in ['madrs', 'hads_d', 'hads_a', 'asrs', 'mdq_pos', 'wurs'] if c in D.columns]
print('\nclinical scale coverage by group:')
print(D.groupby('group')[SCALES].apply(lambda g: g.notna().mean()).round(2))
print('\nbipolar/unipolar flag coverage (per person):')
print(D.drop_duplicates('person').groupby('group')[['bipolar', 'unipolar']]
       .apply(lambda g: g.notna().mean()).round(2))

# ---- 6.2b validate the recipe against the published derived table
hits = {}
for p in sorted(pdf.loc[pdf.group == 'control', 'person'].unique())[:5]:
    mine = np.round(pdf.loc[pdf.person == p, 'mean'].values, 3)
    theirs = np.round(feat.loc[feat['user'] == p, 'mean'].values, 3)
    hits[p] = float(np.isin(theirs, mine).mean()) if len(theirs) else None
S6['recipe_validation_membership'] = hits
print('\nfraction of published values reproduced, per control participant:', hits)
RESULTS['S6'] = S6; save_results()

### 6.3 Benchmark — what the data supports

Real diagnoses from the group, not cluster labels. Grouped by person, with the random
person-day split alongside to size the leakage grouping removes.


In [ ]:
#@title 6.3 · Benchmark
y_g = pd.factorize(D['group'])[0]
S6['diagnosis_5class_subject_grouped'] = cv_f1(D[ACT].values, y_g, groups=D['person'].values)
S6['diagnosis_5class_random_rows']     = cv_f1(D[ACT].values, y_g)
m3 = D['group'].isin(['control', 'depression', 'schizophrenia'])
S6['diagnosis_3class_subject_grouped'] = cv_f1(D.loc[m3, ACT].values,
                                               pd.factorize(D.loc[m3, 'group'])[0],
                                               groups=D.loc[m3, 'person'].values)
for k, floor in [('diagnosis_5class_subject_grouped', 0.200),
                 ('diagnosis_5class_random_rows', 0.200),
                 ('diagnosis_3class_subject_grouped', 0.333)]:
    print(f'{k:36s} {fmt(*S6[k])}   (chance {floor})')
RESULTS['S6'] = S6; save_results()

### 6.4–6.5 Positive control: does the within-class test fire on a real pairing?

The ADHD and clinical groups have same-person actigraphy *and* clinical scales, plus an
explicit `bipolar` flag. This is the fusion design the original pipeline attempted, but with
rows that are real people. If the pairing carries cross-modal information, the within-class
shuffle should cost performance.


In [ ]:
#@title 6.4 · OBF positive control
P_ = D[D['group'].isin(['adhd', 'clinical'])]
per = (P_.groupby('person')
         .agg({**{c: 'mean' for c in ACT}, **{c: 'first' for c in SCALES + ['bipolar', 'group']}})
         .reset_index()
         .dropna(subset=SCALES + ['bipolar']))
y_bp = per['bipolar'].astype(int).values
print(f'{len(per)} people | bipolar={int(y_bp.sum())} non-bipolar={int((1 - y_bp).sum())}')

if len(per) >= 30 and len(np.unique(y_bp)) == 2 and np.bincount(y_bp).min() >= 5:
    A_ = per[ACT].values.astype(float); C_ = per[SCALES].values.astype(float)
    A_ = (A_ - A_.mean(0)) / (A_.std(0) + 1e-9)
    C_ = (C_ - C_.mean(0)) / (C_.std(0) + 1e-9)
    X_ = np.hstack([A_, C_])
    rng = np.random.default_rng(SEED); reps = 10
    res = {'actigraphy_only': cv_f1(A_, y_bp, n_repeats=reps)[0],
           'scales_only':     cv_f1(C_, y_bp, n_repeats=reps)[0],
           'fusion_aligned':  cv_f1(X_, y_bp, n_repeats=reps)[0]}
    wi = [cv_f1(np.hstack([A_, C_[perm_index(len(y_bp), rng, y_bp)]]), y_bp, n_repeats=1, seed=SEED + i)[0]
          for i in range(reps)]
    ac = [cv_f1(np.hstack([A_, C_[perm_index(len(y_bp), rng)]]), y_bp, n_repeats=1, seed=SEED + i)[0]
          for i in range(reps)]
    res['fusion_within_class_shuffle'] = (float(np.mean(wi)), float(np.std(wi)))
    res['fusion_across_class_shuffle'] = (float(np.mean(ac)), float(np.std(ac)))
    S6['positive_control_obf'] = res
    for k, v in res.items():
        print(f'  {k:30s}', fmt(*v) if isinstance(v, tuple) else f'{v:.3f}')
    print(f"  within-class gap: {res['fusion_aligned'] - res['fusion_within_class_shuffle'][0]:+.3f}")
else:
    print('skipped: too few people or classes'); S6['positive_control_obf'] = None
RESULTS['S6'] = S6; save_results()

### 6.6 Second positive control: co-registered WESAD sensors

Chest and wrist are recorded from the same person at the same second, so this is the
strongest available real pairing.

**The permutation must run within (subject, class) cells.** Permuting across all rows under
grouped CV hands a test row its second modality from a *training* subject, which reintroduces
the subject leakage the grouping removed — the shuffle then *improves* the score, which is a
leakage signature and not evidence of alignment.


In [ ]:
#@title 6.6 · WESAD positive control
def perm_within_cells(y, g, rng):
    """Permute rows within each (group, class) cell, so grouping survives the shuffle."""
    idx = np.arange(len(y))
    key = np.array([f'{a}|{b}' for a, b in zip(np.asarray(g), np.asarray(y))])
    for k in np.unique(key):            # sorted, so iteration order is fixed
        pos = np.where(key == k)[0]
        idx[pos] = rng.permutation(pos)
    return idx

chest = [c for c in WESAD_FEATURES if c.startswith('chest')]
wrist = [c for c in WESAD_FEATURES if c.startswith('wrist')]
w = wesad_all[wesad_all.label.isin([1, 2, 3, 4])]
rng_ = np.random.default_rng(SEED)
s = np.concatenate([rng_.choice(np.where(w.label.values == c)[0], 1500, replace=False)
                    for c in [1, 2, 3, 4]])
Aw, Cw = w.iloc[s][chest].values, w.iloc[s][wrist].values
yw, gw = pd.factorize(w.iloc[s].label)[0], w.iloc[s].subject.values

pw = perm_within_cells(yw, gw, rng_)
S6['wesad_chest_wrist_aligned']    = cv_f1(np.hstack([Aw, Cw]),     yw, groups=gw, n_repeats=3)
S6['wesad_chest_wrist_withincell'] = cv_f1(np.hstack([Aw, Cw[pw]]), yw, groups=gw, n_repeats=3)
print('aligned          ', fmt(*S6['wesad_chest_wrist_aligned']))
print('within-cell      ', fmt(*S6['wesad_chest_wrist_withincell']))
print(f"gap: {S6['wesad_chest_wrist_aligned'][0] - S6['wesad_chest_wrist_withincell'][0]:+.3f}")

# NB: the naive across-rows permutation under grouped CV inflates rather than degrades.
pa = perm_index(len(yw), rng_, yw)
S6['wesad_chest_wrist_naive_shuffle_LEAKY'] = cv_f1(np.hstack([Aw, Cw[pa]]), yw, groups=gw, n_repeats=3)
print('naive across-rows shuffle (leaky, for the record):',
      fmt(*S6['wesad_chest_wrist_naive_shuffle_LEAKY']))
RESULTS['S6'] = S6; save_results()

## Save and summarise

In [ ]:
#@title Summary
D.to_csv(f'{OUT_DIR}/obf_corrected_person_day.csv', index=False)
per.to_csv(f'{OUT_DIR}/obf_paired_person_level.csv', index=False)
with open(f'{OUT_DIR}/section6.json', 'w') as f:
    json.dump(S6, f, indent=2, default=lambda o: float(o) if isinstance(o, np.floating) else str(o))

def rnd(o):
    if isinstance(o, dict):  return {str(k): rnd(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [rnd(v) for v in o]
    if isinstance(o, (float, np.floating)): return round(float(o), 3)
    if isinstance(o, np.integer): return int(o)
    return o

lines = ['=== PASTE BACK ===', f"run {RESULTS['meta']['started']}  smoke={SMOKE_TEST}"]
for key in ['A', 'B1_wesad', 'B2_obf', 'B3_text', 'B4_replication', 'B5_leakage',
            'B6_waterfall', 'B7_calibration', 'B8_source_audit', 'B9_join', 'S6']:
    if RESULTS.get(key):
        lines.append(f'{key}: ' + json.dumps(rnd(RESULTS[key]), default=str))
lines.append('=== END ===')
print('\n'.join(lines))
RESULTS['meta']['finished'] = time.strftime('%Y-%m-%d %H:%M'); save_results()
print('\nresults.json  ->', f'{OUT_DIR}/results.json')
print('section6.json ->', f'{OUT_DIR}/section6.json')

In [ ]:
vals = []
for _ in range(2):
    r = np.random.default_rng(SEED)
    r.permutation(len(yw))                      # consume one draw, as 6.6 does
    p = perm_within_cells(yw, gw, np.random.default_rng(SEED))
    vals.append(tuple(p[:20]))
print(vals[0] == vals[1])                       # must be True

In [ ]:
!pip freeze > "{OUT_DIR}/requirements-lock.txt"
print('environment pinned')